# Complete CapsNet + LSTM + LightGBM Pipeline with 5-Fold Cross-Validation

This notebook implements a complete machine learning pipeline that:
1. Loads best hyperparameters from tuning
2. Trains CapsNet and LSTM with 5-fold CV 
3. Extracts features from all folds
4. Fuses features and trains LightGBM
5. Evaluates and compares all 5 fold results

**Author:** Thesis Research  
**Date:** October 2025  
**Purpose:** Air Quality Prediction using Multi-Modal Deep Learning

## 1. Environment Setup and Imports

Import all required libraries and set up the environment for the complete ML pipeline.

In [35]:
import os
import sys
import json
import numpy as np
import pandas as pd
import traceback
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Tuple, Optional
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import lightgbm as lgb
import gc
warnings.filterwarnings('ignore')

# Set up matplotlib for inline plotting
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("📦 All libraries imported successfully!")

# Add src directory to Python path
current_dir = Path.cwd()
src_dir = current_dir / "src"
sys.path.insert(0, str(src_dir))
sys.path.insert(0, str(current_dir))

print(f"📁 Current directory: {current_dir}")
print(f"📁 Source directory: {src_dir}")

# Verify CUDA availability and setup memory optimization
try:
    import torch
    
    # CRITICAL: Memory optimization for 4GB GPU
    # Enable memory-efficient allocator
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    
    # Set to use GPU 0 (your only GPU)
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    
    # Enable deterministic mode to reduce memory overhead
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    
    # Enable memory efficient operations
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    
    if torch.cuda.is_available():
        # Clear any existing GPU cache
        torch.cuda.empty_cache()
        gc.collect()
        
        print(f"🚀 CUDA available! Using GPU 0")
        print(f"🔥 Device: {torch.cuda.get_device_name(0)}")
        print(f"🔥 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        print(f"💾 Memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.3f} GB")
        print(f"💾 Memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.3f} GB")
        print(f"✅ Memory optimization enabled for 4GB GPU")
    else:
        print("⚠️ CUDA not available, using CPU")
except ImportError:
    print("⚠️ PyTorch not found, ensure it's installed for GPU acceleration")

📦 All libraries imported successfully!
📁 Current directory: c:\Users\Micah\Desktop\THESIS
📁 Source directory: c:\Users\Micah\Desktop\THESIS\src
🚀 CUDA available! Using GPU 0
🔥 Device: NVIDIA GeForce RTX 3050 Laptop GPU
🔥 GPU Memory: 4.3 GB
💾 Memory allocated: 0.000 GB
💾 Memory reserved: 0.000 GB
✅ Memory optimization enabled for 4GB GPU


In [36]:
# Import custom model classes
import importlib
try:
    # Force reload to get latest code changes
    if 'src.training.capsnet_trainer' in sys.modules:
        importlib.reload(sys.modules['src.training.capsnet_trainer'])
    if 'src.lstm.lstm_temporal_feature_generator' in sys.modules:
        importlib.reload(sys.modules['src.lstm.lstm_temporal_feature_generator'])
    
    from src.training.capsnet_trainer import CapsNetTrainer, AirQualityDataset
    from src.lstm.lstm_temporal_feature_generator import LSTMTemporalFeatureGenerator
    print("✅ Custom model classes imported successfully!")
    print("✅ Modules reloaded with latest code changes!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("⚠️ Please ensure all model files are in the correct locations")
    print("📝 Trying alternative import paths...")
    try:
        # Try without the src prefix (if src is in sys.path)
        import sys
        from training.capsnet_trainer import CapsNetTrainer, AirQualityDataset
        from lstm.lstm_temporal_feature_generator import LSTMTemporalFeatureGenerator
        print("✅ Custom model classes imported successfully (alternative path)!")
    except ImportError as e2:
        print(f"❌ Alternative import also failed: {e2}")
        print("⚠️ Please check that:")
        print("   1. src/training/capsnet_trainer.py exists")
        print("   2. src/lstm/lstm_temporal_feature_generator.py exists")
        print("   3. All __init__.py files are present in the directories")

✅ Optuna available for hyperparameter tuning
✅ Custom model classes imported successfully!
✅ Modules reloaded with latest code changes!


## 2. Pipeline Configuration

Define the CompleteMLPipeline class with all necessary methods for the end-to-end machine learning pipeline.

In [37]:
class CompleteMLPipeline:
    """Complete ML Pipeline with CapsNet + LSTM + LightGBM"""
    
    def __init__(self, day_folder: str, output_dir: str = "pipeline_outputs", fast_mode: bool = False):
        self.day_folder = day_folder
        self.output_dir = output_dir
        self.n_folds = 2 if fast_mode else 5  # 2 folds for quick testing, 5 for full CV
        self.fast_mode = fast_mode
        self.device = 'cuda:0'  # Using GPU 0 (your only GPU)
        
        # Clear GPU memory before initialization
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                gc.collect()
                print(f"🧹 GPU memory cleared before pipeline initialization")
        except:
            pass
        
        # Create organized output directories
        self.setup_directories()
        
        # Results storage
        self.fold_results = []
        self.capsnet_features = {}
        self.lstm_features = {}
        self.final_results = {}
        
        print(f"🚀 Complete ML Pipeline initialized for {day_folder}")
        print(f"📁 Output directory: {output_dir}")
        print(f"🔄 Using {self.n_folds}-fold cross-validation")
        print(f"🎮 Using device: {self.device} (GPU 0 - RTX 3050 4GB)")
        if fast_mode:
            print(f"⚡ FAST MODE: 2-fold CV for quick testing")
        else:
            print(f"📊 FULL MODE: 5-fold cross-validation for robust evaluation")
    
    def setup_directories(self):
        """Create organized directory structure"""
        directories = [
            self.output_dir,
            f"{self.output_dir}/models/capsnet",
            f"{self.output_dir}/models/lstm", 
            f"{self.output_dir}/models/lightgbm",
            f"{self.output_dir}/features/capsnet",
            f"{self.output_dir}/features/lstm",
            f"{self.output_dir}/features/fused",
            f"{self.output_dir}/results",
            f"{self.output_dir}/plots",
            f"{self.output_dir}/cv_folds"
        ]
        
        for directory in directories:
            os.makedirs(directory, exist_ok=True)
        
        print(f"✅ Directory structure created!")

print("✅ CompleteMLPipeline class defined!")

✅ CompleteMLPipeline class defined!


## 3. Data Loading and Hyperparameter Setup

Load best hyperparameters from previous tuning results and set up default parameters.

In [38]:
# Add hyperparameter loading method to the pipeline class
def load_best_hyperparameters(self, model_type: str) -> Dict:
    """Load best hyperparameters from previous tuning results"""
    print(f"📋 Loading best hyperparameters for {model_type}...")
    
    # Look for hyperparameter files
    import glob
    
    # Extract the date part from day_folder (e.g., "7_24_data" -> "7_24")
    date_part = self.day_folder.replace('_data', '') if '_data' in self.day_folder else self.day_folder
    
    # Define different patterns for different model types
    if model_type == "capsnet":
        param_patterns = [
            f"outputs/capsnet/hyperparameters/basic/best_params_basic_{self.day_folder}_*.json",
            f"outputs/capsnet/hyperparameters/advanced/best_params_advanced_{self.day_folder}_*.json",
            f"best_params_capsnet_{self.day_folder}.json",
            f"best_params_capsnet.json"
        ]
    elif model_type == "lstm":
        param_patterns = [
            f"src/lstm/{date_part}_best_params.json",  # Matches: 7_24_best_params.json
            f"src/lstm/{self.day_folder}_best_params.json",  # Alternative: 7_24_data_best_params.json
            f"src/lstm/best_params_{date_part}.json",  # Another format: best_params_7_24.json
            f"outputs/lstm/hyperparameters/best_params_{self.day_folder}_*.json",  # Fallback
            f"best_params_lstm_{self.day_folder}.json",
            f"best_params_lstm.json"
        ]
    else:
        param_patterns = [
            f"best_params_{model_type}_{self.day_folder}.json",
            f"best_params_{model_type}.json"
        ]
    
    best_params = None
    for pattern in param_patterns:
        files = glob.glob(pattern)
        if files:
            # Use the most recent file
            latest_file = max(files, key=os.path.getmtime)
            try:
                with open(latest_file, 'r') as f:
                    best_params = json.load(f)
                print(f"   ✅ Loaded parameters from: {latest_file}")
                break
            except Exception as e:
                print(f"   ⚠️ Error loading {latest_file}: {e}")
                continue
    
    if best_params is None:
        print(f"   ⚠️ No saved hyperparameters found for {model_type}, using defaults")
        # Default parameters
        if model_type == "capsnet":
            best_params = {
                'learning_rate': 0.001,
                'dropout_rate': 0.3,
                'feature_dim': 128,
                'optimizer_type': 'adam',
                'weight_decay': 0.0001,
                'batch_size': 8
            }
        elif model_type == "lstm":
            best_params = {
                'learning_rate': 0.001,
                'hidden_size': 128,
                'num_layers': 2,
                'dropout': 0.2,
                'batch_size': 32
            }
    
    print(f"   📊 {model_type.upper()} parameters: {best_params}")
    return best_params

# Add the method to the class
CompleteMLPipeline.load_best_hyperparameters = load_best_hyperparameters
print("✅ Hyperparameter loading method added to pipeline class!")

✅ Hyperparameter loading method added to pipeline class!


## 4. CapsNet Cross-Validation Training

Implement 5-fold cross-validation training for the CapsNet model.

In [39]:
def train_capsnet_cv(self, capsnet_params: Dict) -> Dict[int, str]:
    """Train CapsNet with 5-fold cross-validation"""
    print(f"\n🔄 Training CapsNet with {self.n_folds}-fold CV...")
    print(f"   Parameters: {capsnet_params}")
    print(f"   ⚡ Using SimplifiedCapsNet") #for 4GB GPU compatibility
    
    # Clear GPU memory before training
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
            print(f"🧹 GPU memory cleared before training")
            print(f"💾 Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    except:
        pass
    
    # Initialize trainer with GPU 0
    trainer = CapsNetTrainer(
        input_size=256,
        feature_dim=capsnet_params.get('feature_dim', 128),
        device=self.device  # Use GPU 0
    )
    
    # Load data
    learning_df, patch_metadata_df = trainer.load_day_data(self.day_folder)
    
    # Setup K-fold CV
    kfold = KFold(n_splits=self.n_folds, shuffle=True, random_state=42)
    fold_models = {}
    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(learning_df)):
        print(f"\n📊 CapsNet Fold {fold + 1}/{self.n_folds}")
        print("-" * 40)
        
        # Clear GPU memory before each fold
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                gc.collect()
                mem_allocated = torch.cuda.memory_allocated(0) / 1e9
                mem_reserved = torch.cuda.memory_reserved(0) / 1e9
                print(f"🧹 GPU memory cleared for fold {fold+1}")
                print(f"💾 Allocated: {mem_allocated:.3f} GB | Reserved: {mem_reserved:.3f} GB")
        except:
            pass
        
        # Split data for this fold
        train_df = learning_df.iloc[train_idx].reset_index(drop=True)
        val_df = learning_df.iloc[val_idx].reset_index(drop=True)
        
        # Create datasets
        train_dataset = AirQualityDataset(train_df, patch_metadata_df, self.day_folder, 'train')
        val_dataset = AirQualityDataset(val_df, patch_metadata_df, self.day_folder, 'val')
        
        print(f"   Training samples: {len(train_dataset)}")
        print(f"   Validation samples: {len(val_dataset)}")
        
        # Create model for this fold
        # Filter out parameters that are already passed to __init__ or create_model directly
        model_params = {k: v for k, v in capsnet_params.items() 
                       if k not in ['batch_size', 'feature_dim', 'learning_rate', 'weight_decay', 'optimizer_type']}
        
        print(f"🔧 Creating SimplifiedCapsNet model for fold {fold+1}...")
        # Use simplified=True for 4GB GPU
        trainer.create_model(use_simplified=True, **model_params)
        
        trainer.setup_training(
            learning_rate=capsnet_params.get('learning_rate', 0.001),
            weight_decay=capsnet_params.get('weight_decay', 1e-4),
            optimizer_type=capsnet_params.get('optimizer_type', 'adam')
        )
        
        # Train (adaptive epochs based on mode)
        epochs = 10 if self.fast_mode else 15
        best_loss = trainer.train(
            train_dataset, val_dataset,
            epochs=epochs,
            batch_size=capsnet_params.get('batch_size', 8),  # Use batch size from params
            day_folder=f"{self.day_folder}_fold_{fold+1}"
        )
        
        # Save fold model
        fold_model_path = f"{self.output_dir}/models/capsnet/capsnet_simplified_fold_{fold+1}_{self.day_folder}.pth"
        trainer.save_model(fold_model_path, 30, best_loss)
        fold_models[fold+1] = fold_model_path
        
        print(f"   ✅ Fold {fold+1} completed! Best loss: {best_loss:.4f}")
        
        # Clear memory after fold
        try:
            import torch
            if torch.cuda.is_available():
                del trainer.model
                torch.cuda.empty_cache()
                gc.collect()
                print(f"🧹 Memory cleared after fold {fold+1}")
        except:
            pass
    
    print(f"\n✅ CapsNet {self.n_folds}-fold CV completed!")
    return fold_models

# Add the method to the class
CompleteMLPipeline.train_capsnet_cv = train_capsnet_cv
print("✅ CapsNet cross-validation training method added (using SimplifiedCapsNet)!")

✅ CapsNet cross-validation training method added (using SimplifiedCapsNet)!


## 5. LSTM Cross-Validation Training

Implement 5-fold cross-validation training for the LSTM model.

In [40]:
def train_lstm_cv(self, lstm_params: Dict) -> Dict[int, str]:
    """Train LSTM with 5-fold expanding window cross-validation and extract features"""
    print(f"\n🔄 Training LSTM with {self.n_folds}-fold expanding window CV...")
    print(f"   Parameters: {lstm_params}")

    # Load temporal data and targets
    from src.lstm.lstm_temporal_feature_generator import LSTMTemporalFeatureGenerator, TemporalDataLoader
    data_loader = TemporalDataLoader()
    
    # Extract date part from day_folder (e.g., "7_24_data" -> "7_24")
    date_part = self.day_folder.replace('_data', '') if '_data' in self.day_folder else self.day_folder
    temporal_data, targets, feature_names = data_loader.load_temporal_data(date_part)

    n_total = len(temporal_data)
    n_learning = int(n_total * 0.8)
    learning_temporal = temporal_data[:n_learning]
    learning_targets = targets[:n_learning]

    from sklearn.model_selection import TimeSeriesSplit
    tscv = TimeSeriesSplit(n_splits=self.n_folds)
    lstm_generator = LSTMTemporalFeatureGenerator(lstm_params)

    fold_models = {}
    for fold, (train_idx, val_idx) in enumerate(tscv.split(learning_temporal)):
        print(f"\n📊 LSTM Fold {fold + 1}/{self.n_folds}")
        print("-" * 40)

        train_temporal = learning_temporal[train_idx]
        train_targets = learning_targets[train_idx]
        val_temporal = learning_temporal[val_idx]
        val_targets = learning_targets[val_idx]

        # Train LSTM and extract features
        train_temp_features, val_temp_features, _, _ = lstm_generator.train_and_extract_features(
            train_temporal, train_targets, val_temporal, val_targets
        )

        # Save features for this fold
        features_path = f"{self.output_dir}/features/lstm_fold_{fold+1}_{self.day_folder}.npz"
        np.savez(features_path,
                 train_features=train_temp_features,
                 val_features=val_temp_features,
                 train_targets=train_targets,
                 val_targets=val_targets)
        fold_models[fold+1] = features_path

        print(f"   ✅ LSTM Fold {fold+1} completed! Features saved to {features_path}")

    print(f"\n✅ LSTM {self.n_folds}-fold CV completed!")
    return fold_models

# Add the method to the class
CompleteMLPipeline.train_lstm_cv = train_lstm_cv
print("✅ LSTM cross-validation training method updated!")

✅ LSTM cross-validation training method updated!


## 6. Feature Extraction from All Folds

Extract features from trained CapsNet and LSTM models for each cross-validation fold.

In [41]:
def extract_features_cv(self, capsnet_models: Dict[int, str], 
                      lstm_models: Dict[int, str]) -> Tuple[Dict, Dict]:
    """Extract features from all CV folds"""
    print(f"\n🔍 Extracting features from all CV folds...")
    
    capsnet_features = {}
    lstm_features = {}
    
    for fold in range(1, self.n_folds + 1):
        print(f"\n📊 Extracting features from Fold {fold}")
        
        # Extract CapsNet features
        print(f"   🔍 CapsNet features...")
        capsnet_features[fold] = self.extract_capsnet_features_fold(
            capsnet_models[fold], fold
        )
        
        # Extract LSTM features  
        print(f"   🔍 LSTM features...")
        lstm_features[fold] = self.extract_lstm_features_fold(
            lstm_models[fold], fold
        )
        
        print(f"   ✅ Fold {fold} features extracted!")
    
    self.capsnet_features = capsnet_features
    self.lstm_features = lstm_features
    
    print(f"\n✅ All features extracted!")
    return capsnet_features, lstm_features

def extract_capsnet_features_fold(self, model_path: str, fold: int) -> pd.DataFrame:
    """Extract CapsNet features for a specific fold"""
    # Initialize trainer with GPU 1
    trainer = CapsNetTrainer(input_size=256, feature_dim=128, device=self.device)
    trainer.create_model()
    trainer.load_model(model_path)
    
    # Load data for this fold
    learning_df, patch_metadata_df = trainer.load_day_data(self.day_folder)
    
    # For CV, we need to recreate the same split
    kfold = KFold(n_splits=self.n_folds, shuffle=True, random_state=42)
    splits = list(kfold.split(learning_df))
    train_idx, val_idx = splits[fold-1]
    
    # Use validation set for feature extraction
    val_df = learning_df.iloc[val_idx].reset_index(drop=True)
    
    val_dataset = AirQualityDataset(val_df, patch_metadata_df, self.day_folder, 'val')
    
    # Extract features
    features, metadata = trainer.extract_features(
        val_dataset, 
        day_folder=f"{self.day_folder}_fold_{fold}",
        split_name=f'fold_{fold}'
    )
    
    # Convert to DataFrame
    feature_df = pd.DataFrame(features, columns=[f'capsnet_f_{i}' for i in range(len(features[0]))])
    
    # Add metadata
    if metadata:
        for key in ['image_filename', 'timestamp', 'pm2.5']:
            if key in metadata[0]:
                feature_df[key] = [m[key] for m in metadata]
    
    # Save features
    feature_path = f"{self.output_dir}/features/capsnet/capsnet_features_fold_{fold}_{self.day_folder}.csv"
    feature_df.to_csv(feature_path, index=False)
    
    return feature_df

def extract_lstm_features_fold(self, model_path: str, fold: int) -> pd.DataFrame:
    """Extract LSTM features for a specific fold (placeholder)"""
    # This is a placeholder - implement your actual LSTM feature extraction
    
    # Load LSTM data
    try:
        lstm_data = pd.read_csv(f"dataset/lstm_features_{self.day_folder}.csv")
    except FileNotFoundError:
        # Create dummy LSTM features
        learning_df = pd.read_csv(f"dataset/d_data_split/{self.day_folder}/learning.csv")
        lstm_data = pd.DataFrame({
            'timestamp': learning_df['timestamp'],
            'pm2.5': learning_df['pm2.5'],
            **{f'lstm_f_{i}': np.random.randn(len(learning_df)) for i in range(64)}
        })
    
    # For CV, recreate the same split
    kfold = KFold(n_splits=self.n_folds, shuffle=True, random_state=42)
    splits = list(kfold.split(lstm_data))
    train_idx, val_idx = splits[fold-1]
    
    # Use validation set
    val_features = lstm_data.iloc[val_idx].reset_index(drop=True)
    
    # Save features
    feature_path = f"{self.output_dir}/features/lstm/lstm_features_fold_{fold}_{self.day_folder}.csv"
    val_features.to_csv(feature_path, index=False)
    
    return val_features

# Add the methods to the class
CompleteMLPipeline.extract_features_cv = extract_features_cv
CompleteMLPipeline.extract_capsnet_features_fold = extract_capsnet_features_fold
CompleteMLPipeline.extract_lstm_features_fold = extract_lstm_features_fold
print("✅ Feature extraction methods added!")

✅ Feature extraction methods added!


## 7. Feature Fusion and LightGBM Training

Fuse CapsNet and LSTM features for each fold and train LightGBM models.

In [42]:
def fuse_features_and_train_lightgbm(self) -> Dict[int, Dict]:
    """Fuse features from all folds and train LightGBM"""
    print(f"\n🤝 Fusing features and training LightGBM for all folds...")
    
    fold_results = {}
    
    for fold in range(1, self.n_folds + 1):
        print(f"\n📊 Processing Fold {fold}")
        print("-" * 40)
        
        # Load features for this fold
        capsnet_df = self.capsnet_features[fold]
        lstm_df = self.lstm_features[fold]
        
        # Fuse features
        print("   🤝 Fusing CapsNet and LSTM features...")
        fused_features = self.fuse_features_fold(capsnet_df, lstm_df, fold)
        
        # Train LightGBM
        print("   🚀 Training LightGBM...")
        fold_result = self.train_lightgbm_fold(fused_features, fold)
        fold_results[fold] = fold_result
        
        print(f"   ✅ Fold {fold} LightGBM training completed!")
        print(f"       RMSE: {fold_result['rmse']:.4f}")
        print(f"       MAE: {fold_result['mae']:.4f}")  
        print(f"       R²: {fold_result['r2']:.4f}")
    
    self.fold_results = fold_results
    print(f"\n✅ All LightGBM models trained!")
    return fold_results

def fuse_features_fold(self, capsnet_df: pd.DataFrame, lstm_df: pd.DataFrame, 
                      fold: int) -> pd.DataFrame:
    """Fuse CapsNet and LSTM features for a specific fold"""
    
    # Align dataframes by timestamp if available
    if 'timestamp' in capsnet_df.columns and 'timestamp' in lstm_df.columns:
        # Merge on timestamp
        fused_df = pd.merge(capsnet_df, lstm_df, on='timestamp', suffixes=('_capsnet', '_lstm'))
    else:
        # Simple concatenation if timestamps don't align
        min_len = min(len(capsnet_df), len(lstm_df))
        capsnet_features = capsnet_df.iloc[:min_len]
        lstm_features = lstm_df.iloc[:min_len]
        
        # Combine features
        fused_df = pd.concat([
            capsnet_features.reset_index(drop=True),
            lstm_features.reset_index(drop=True)
        ], axis=1)
    
    # Use pm2.5 from CapsNet (more reliable)
    if 'pm2.5_capsnet' in fused_df.columns:
        fused_df['pm2.5'] = fused_df['pm2.5_capsnet']
    elif 'pm2.5_lstm' in fused_df.columns:
        fused_df['pm2.5'] = fused_df['pm2.5_lstm']
    
    # Save fused features
    fused_path = f"{self.output_dir}/features/fused/fused_features_fold_{fold}_{self.day_folder}.csv"
    fused_df.to_csv(fused_path, index=False)
    
    print(f"       Fused features shape: {fused_df.shape}")
    print(f"       CapsNet features: {len([c for c in fused_df.columns if 'capsnet_f_' in c])}")
    print(f"       LSTM features: {len([c for c in fused_df.columns if 'lstm_f_' in c])}")
    
    return fused_df

def train_lightgbm_fold(self, fused_df: pd.DataFrame, fold: int) -> Dict:
    """Train LightGBM for a specific fold"""
    
    # Prepare features and target
    feature_cols = [c for c in fused_df.columns if c.startswith(('capsnet_f_', 'lstm_f_'))]
    X = fused_df[feature_cols]
    y = fused_df['pm2.5']
    
    # Remove any NaN values
    mask = ~(X.isna().any(axis=1) | y.isna())
    X = X[mask]
    y = y[mask]
    
    print(f"       Training samples: {len(X)}")
    print(f"       Feature columns: {len(feature_cols)}")
    
    # Split for training/validation within fold
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # LightGBM parameters
    lgb_params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'random_state': 42
    }
    
    # Create datasets
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    # Train model
    model = lgb.train(
        lgb_params,
        train_data,
        valid_sets=[val_data],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )
    
    # Make predictions
    y_pred = model.predict(X_val)
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    
    # Save model
    model_path = f"{self.output_dir}/models/lightgbm/lightgbm_fold_{fold}_{self.day_folder}.txt"
    model.save_model(model_path)
    
    return {
        'fold': fold,
        'model_path': model_path,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'feature_importance': dict(zip(feature_cols, model.feature_importance())),
        'predictions': y_pred,
        'actual': y_val.values
    }

# Add the methods to the class
CompleteMLPipeline.fuse_features_and_train_lightgbm = fuse_features_and_train_lightgbm
CompleteMLPipeline.fuse_features_fold = fuse_features_fold
CompleteMLPipeline.train_lightgbm_fold = train_lightgbm_fold
print("✅ Feature fusion and LightGBM training methods added!")

✅ Feature fusion and LightGBM training methods added!


## 8. Results Analysis and Metrics

Analyze cross-validation results across all folds and calculate comprehensive metrics.

In [43]:
def analyze_results(self) -> Dict:
    """Analyze and compare results across all folds"""
    print(f"\n📊 Analyzing results across all {self.n_folds} folds...")
    
    # Collect metrics
    fold_metrics = []
    for fold, result in self.fold_results.items():
        fold_metrics.append({
            'fold': fold,
            'rmse': result['rmse'],
            'mae': result['mae'],
            'r2': result['r2']
        })
    
    metrics_df = pd.DataFrame(fold_metrics)
    
    # Calculate statistics
    stats = {
        'mean_rmse': metrics_df['rmse'].mean(),
        'std_rmse': metrics_df['rmse'].std(),
        'mean_mae': metrics_df['mae'].mean(),
        'std_mae': metrics_df['mae'].std(),
        'mean_r2': metrics_df['r2'].mean(),
        'std_r2': metrics_df['r2'].std(),
        'best_fold': metrics_df.loc[metrics_df['rmse'].idxmin(), 'fold'],
        'worst_fold': metrics_df.loc[metrics_df['rmse'].idxmax(), 'fold']
    }
    
    self.final_results = {
        'fold_metrics': fold_metrics,
        'statistics': stats,
        'day_folder': self.day_folder
    }
    
    # Print results
    print(f"\n🎯 Cross-Validation Results Summary:")
    print(f"   Average RMSE: {stats['mean_rmse']:.4f} ± {stats['std_rmse']:.4f}")
    print(f"   Average MAE:  {stats['mean_mae']:.4f} ± {stats['std_mae']:.4f}")
    print(f"   Average R²:   {stats['mean_r2']:.4f} ± {stats['std_r2']:.4f}")
    print(f"   Best fold:    {stats['best_fold']} (RMSE: {metrics_df.loc[stats['best_fold']-1, 'rmse']:.4f})")
    print(f"   Worst fold:   {stats['worst_fold']} (RMSE: {metrics_df.loc[stats['worst_fold']-1, 'rmse']:.4f})")
    
    # Save results
    results_path = f"{self.output_dir}/results/cv_results_{self.day_folder}.json"
    with open(results_path, 'w') as f:
        json.dump(self.final_results, f, indent=2, default=str)
    
    metrics_path = f"{self.output_dir}/results/fold_metrics_{self.day_folder}.csv"
    metrics_df.to_csv(metrics_path, index=False)
    
    print(f"   💾 Results saved to: {results_path}")
    print(f"   💾 Metrics saved to: {metrics_path}")
    
    return self.final_results

# Add the method to the class
CompleteMLPipeline.analyze_results = analyze_results
print("✅ Results analysis method added!")

✅ Results analysis method added!


## 9. Visualization Creation

Create comprehensive visualizations for model evaluation and results interpretation.

In [44]:
def create_visualizations(self):
    """Create visualizations for the results"""
    print(f"\n📈 Creating visualizations...")
    
    # 1. Fold comparison plot
    self.plot_fold_comparison()
    
    # 2. Feature importance plot
    self.plot_feature_importance()
    
    # 3. Predictions vs actual plot
    self.plot_predictions_vs_actual()
    
    print(f"   💾 Visualizations saved to: {self.output_dir}/plots/")

def plot_fold_comparison(self):
    """Plot comparison of metrics across folds"""
    metrics_data = []
    for fold, result in self.fold_results.items():
        metrics_data.extend([
            {'fold': fold, 'metric': 'RMSE', 'value': result['rmse']},
            {'fold': fold, 'metric': 'MAE', 'value': result['mae']},
            {'fold': fold, 'metric': 'R²', 'value': result['r2']}
        ])
    
    metrics_df = pd.DataFrame(metrics_data)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    for i, metric in enumerate(['RMSE', 'MAE', 'R²']):
        data = metrics_df[metrics_df['metric'] == metric]
        axes[i].bar(data['fold'], data['value'], alpha=0.7)
        axes[i].set_title(f'{metric} by Fold')
        axes[i].set_xlabel('Fold')
        axes[i].set_ylabel(metric)
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{self.output_dir}/plots/fold_comparison_{self.day_folder}.png", dpi=300, bbox_inches='tight')
    plt.show()

def plot_feature_importance(self):
    """Plot feature importance across folds"""
    # Aggregate feature importance across folds
    all_importance = {}
    for fold, result in self.fold_results.items():
        for feature, importance in result['feature_importance'].items():
            if feature not in all_importance:
                all_importance[feature] = []
            all_importance[feature].append(importance)
    
    # Calculate mean importance
    mean_importance = {k: np.mean(v) for k, v in all_importance.items()}
    
    # Sort by importance
    sorted_features = sorted(mean_importance.items(), key=lambda x: x[1], reverse=True)
    
    # Plot top 20 features
    top_features = sorted_features[:20]
    features, importance = zip(*top_features)
    
    plt.figure(figsize=(12, 8))
    plt.barh(range(len(features)), importance, alpha=0.7)
    plt.yticks(range(len(features)), features)
    plt.xlabel('Feature Importance')
    plt.title(f'Top 20 Feature Importance - {self.day_folder}')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{self.output_dir}/plots/feature_importance_{self.day_folder}.png", dpi=300, bbox_inches='tight')
    plt.show()

def plot_predictions_vs_actual(self):
    """Plot predictions vs actual values for all folds"""
    n_cols = 3
    n_rows = (self.n_folds + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
    
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for fold, result in self.fold_results.items():
        row = (fold - 1) // n_cols
        col = (fold - 1) % n_cols
        ax = axes[row, col]
        
        actual = result['actual']
        pred = result['predictions']
        
        # Scatter plot
        ax.scatter(actual, pred, alpha=0.6)
        
        # Perfect prediction line
        min_val = min(actual.min(), pred.min())
        max_val = max(actual.max(), pred.max())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8)
        
        ax.set_xlabel('Actual PM2.5')
        ax.set_ylabel('Predicted PM2.5')
        ax.set_title(f'Fold {fold} - R² = {result["r2"]:.4f}')
        ax.grid(True, alpha=0.3)
    
    # Remove empty subplots if any
    for i in range(self.n_folds, n_rows * n_cols):
        row = i // n_cols
        col = i % n_cols
        fig.delaxes(axes[row, col])
    
    plt.tight_layout()
    plt.savefig(f"{self.output_dir}/plots/predictions_vs_actual_{self.day_folder}.png", dpi=300, bbox_inches='tight')
    plt.show()

# Add the methods to the class
CompleteMLPipeline.create_visualizations = create_visualizations
CompleteMLPipeline.plot_fold_comparison = plot_fold_comparison
CompleteMLPipeline.plot_feature_importance = plot_feature_importance
CompleteMLPipeline.plot_predictions_vs_actual = plot_predictions_vs_actual
print("✅ Visualization methods added!")

✅ Visualization methods added!


## 10. Main Pipeline Execution

Complete pipeline execution method that orchestrates all the components.

In [45]:
def run_complete_pipeline(self) -> Dict:
    """Run the complete pipeline"""
    print(f"🚀 Starting Complete ML Pipeline for {self.day_folder}")
    print("=" * 60)
    
    try:
        # Step 1: Load hyperparameters
        print(f"\n📋 Step 1: Loading Best Hyperparameters")
        capsnet_params = self.load_best_hyperparameters('capsnet')
        lstm_params = self.load_best_hyperparameters('lstm')
        
        # Step 2: Train models with CV
        print(f"\n📋 Step 2: Training Models with {self.n_folds}-Fold CV")
        lstm_models = self.train_lstm_cv(lstm_params)
        capsnet_models = self.train_capsnet_cv(capsnet_params)
        
        # Step 3: Extract features
        print(f"\n📋 Step 3: Extracting Features from All Folds")
        self.extract_features_cv(capsnet_models, lstm_models)
        
        # Step 4: Fuse features and train LightGBM
        print(f"\n📋 Step 4: Fusing Features and Training LightGBM")
        self.fuse_features_and_train_lightgbm()
        
        # Step 5: Analyze results
        print(f"\n📋 Step 5: Analyzing Results")
        results = self.analyze_results()
        
        # Step 6: Create visualizations
        print(f"\n📋 Step 6: Creating Visualizations")
        self.create_visualizations()
        
        print(f"\n🎉 Complete Pipeline Finished Successfully!")
        print("=" * 60)
        
        return results
        
    except Exception as e:
        print(f"\n❌ Pipeline failed: {e}")
        traceback.print_exc()
        return None

# Add the method to the class
CompleteMLPipeline.run_complete_pipeline = run_complete_pipeline
print("✅ Main pipeline execution method added!")

✅ Main pipeline execution method added!


## 11. Cross-Day Comparison

Functions for running the pipeline across multiple days and creating comparative analysis.

In [46]:
def create_cross_day_comparison(all_results: Dict, output_dir: str):
    """Create comparison plots across different days"""
    comparison_data = []
    
    for day, result in all_results.items():
        if result and 'statistics' in result:
            stats = result['statistics']
            comparison_data.append({
                'day': day,
                'mean_rmse': stats['mean_rmse'],
                'std_rmse': stats['std_rmse'],
                'mean_mae': stats['mean_mae'],
                'std_mae': stats['std_mae'],
                'mean_r2': stats['mean_r2'],
                'std_r2': stats['std_r2']
            })
    
    if comparison_data:
        comparison_df = pd.DataFrame(comparison_data)
        
        # Create comparison plots
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        for i, metric in enumerate(['rmse', 'mae', 'r2']):
            mean_col = f'mean_{metric}'
            std_col = f'std_{metric}'
            
            axes[i].bar(comparison_df['day'], comparison_df[mean_col], 
                       yerr=comparison_df[std_col], alpha=0.7, capsize=5)
            axes[i].set_title(f'{metric.upper()} Comparison Across Days')
            axes[i].set_ylabel(metric.upper())
            axes[i].tick_params(axis='x', rotation=45)
            axes[i].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f"{output_dir}/cross_day_comparison.png", dpi=300, bbox_inches='tight')
        plt.show()
        
        # Save comparison data
        comparison_df.to_csv(f"{output_dir}/cross_day_results.csv", index=False)
        
        print(f"📊 Cross-day comparison saved to: {output_dir}/")
        return comparison_df
    
    return None

def run_pipeline_for_all_days(output_base_dir: str = "pipeline_outputs", fast_mode: bool = False):
    """Run pipeline for all available days"""
    print("🚀 Running Complete Pipeline for All Days")
    print("=" * 60)
    
    days = ['7_24_data', '10_19_data', '11_10_data']
    all_results = {}
    
    for day in days:
        print(f"\n🗓️ Processing {day}...")
        pipeline = CompleteMLPipeline(day, f"{output_base_dir}/{day}", fast_mode=fast_mode)
        result = pipeline.run_complete_pipeline()
        all_results[day] = result
    
    # Create comparison across days
    print(f"\n📊 Creating Cross-Day Comparison...")
    comparison_df = create_cross_day_comparison(all_results, output_base_dir)
    
    return all_results, comparison_df

print("✅ Cross-day comparison functions defined!")

✅ Cross-day comparison functions defined!


## 12. Interactive Pipeline Execution

Now you can run the pipeline interactively! Choose your configuration and execute.

In [47]:
# Configuration
DAY_FOLDER = '7_24_data'  # Change this to: '7_24_data', '10_19_data', or '11_10_data'
OUTPUT_DIR = 'pipeline_outputs'
FAST_MODE = True  # Set to False for full 5-fold CV, True for quick 2-fold testing

print(f"📋 Configuration:")
print(f"   Day folder: {DAY_FOLDER}")
print(f"   Output directory: {OUTPUT_DIR}")
print(f"   Fast mode: {'ON (2-fold CV for quick testing)' if FAST_MODE else 'OFF (5-fold CV for full evaluation)'}")
print(f"   Cross-validation folds: {2 if FAST_MODE else 5}")

# Check if required data exists
import os
learning_data_path = f"dataset/d_data_split/{DAY_FOLDER}/learning.csv"
if os.path.exists(learning_data_path):
    print(f"✅ Learning data found: {learning_data_path}")
else:
    print(f"❌ Learning data not found: {learning_data_path}")
    print("   Please ensure the data preprocessing has been completed")

patch_metadata_path = "dataset/e_preprocessed_img/patch_metadata.csv"
if os.path.exists(patch_metadata_path):
    print(f"✅ Patch metadata found: {patch_metadata_path}")
else:
    print(f"❌ Patch metadata not found: {patch_metadata_path}")
    print("   Please ensure the image preprocessing has been completed")

📋 Configuration:
   Day folder: 7_24_data
   Output directory: pipeline_outputs
   Fast mode: ON (2-fold CV for quick testing)
   Cross-validation folds: 2
✅ Learning data found: dataset/d_data_split/7_24_data/learning.csv
✅ Patch metadata found: dataset/e_preprocessed_img/patch_metadata.csv


In [48]:
# Initialize and run the pipeline for a single day
print(f"\n🚀 Initializing Complete ML Pipeline...")

pipeline = CompleteMLPipeline(
    day_folder=DAY_FOLDER,
    output_dir=OUTPUT_DIR,
    fast_mode=FAST_MODE
)

print(f"\n📊 Pipeline initialized successfully!")
print(f"   Ready to train {pipeline.n_folds}-fold cross-validation")


🚀 Initializing Complete ML Pipeline...
🧹 GPU memory cleared before pipeline initialization
✅ Directory structure created!
🚀 Complete ML Pipeline initialized for 7_24_data
📁 Output directory: pipeline_outputs
🔄 Using 2-fold cross-validation
🎮 Using device: cuda:0 (GPU 0 - RTX 3050 4GB)
⚡ FAST MODE: 2-fold CV for quick testing

📊 Pipeline initialized successfully!
   Ready to train 2-fold cross-validation


In [49]:
# Run the complete pipeline
# This cell will execute the entire pipeline - may take several hours depending on configuration

print("🎯 Starting Complete Pipeline Execution...")
print("⚠️ This may take several hours depending on your configuration")
print("💡 You can monitor progress in the output below")

# Uncomment the line below to run the pipeline
results = pipeline.run_complete_pipeline()

print("📝 Uncomment the line above to execute the pipeline")
print("🔧 Make sure all dependencies are installed and data is preprocessed first")

🎯 Starting Complete Pipeline Execution...
⚠️ This may take several hours depending on your configuration
💡 You can monitor progress in the output below
🚀 Starting Complete ML Pipeline for 7_24_data

📋 Step 1: Loading Best Hyperparameters
📋 Loading best hyperparameters for capsnet...
   ✅ Loaded parameters from: outputs/capsnet/hyperparameters/basic\best_params_basic_7_24_data_20251014_043620_20251013_232346.json
   📊 CAPSNET parameters: {'learning_rate': 0.01, 'dropout_rate': 0.4659969709057026, 'feature_dim': 64, 'optimizer_type': 'adam', 'weight_decay': 0.01, 'batch_size': 4}
📋 Loading best hyperparameters for lstm...
   ✅ Loaded parameters from: src/lstm/7_24_best_params.json
   📊 LSTM parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'activation': 'tanh', 'learning_rate': 0.001421977390625334, 'batch_size': 64, 'epochs': 30, 'timesteps': 60, 'weight_decay': 0.0001, 'grad_clip': 1.0, 'lstm_dropout': 0.1}

📋 Step 2: Training Models with 2-Fold CV

🔄 Training LSTM with 

Mapping patches: 100%|██████████| 28255/28255 [00:29<00:00, 960.42it/s] 



   Final dataset size: 282550 samples
   Expansion factor: 10.0x
📊 Dataset initialization for 7_24_data (val):
   Input learning data: 28255 entries
   Available patches for day: 874 patches


Mapping patches: 100%|██████████| 28255/28255 [00:29<00:00, 947.16it/s] 



   Final dataset size: 282550 samples
   Expansion factor: 10.0x
   Training samples: 282550
   Validation samples: 282550
🔧 Creating SimplifiedCapsNet model for fold 1...
[CapsNetTrainer] Using SimplifiedCapsNet for feature extraction.
   Total parameters: 4,902,721
🚀 Starting CapsNet training...
   Epochs: 10
   Batch size: 4
   Training samples: 282550
   Validation samples: 282550
📋 Experiment info saved: outputs/capsnet/experiments\runs\experiment_7_24_data_fold_1_20251022_131153.json

📊 Epoch 1/10
--------------------------------------------------
[CapsNetTrainer] Using SimplifiedCapsNet for feature extraction.
   Total parameters: 4,902,721
🚀 Starting CapsNet training...
   Epochs: 10
   Batch size: 4
   Training samples: 282550
   Validation samples: 282550
📋 Experiment info saved: outputs/capsnet/experiments\runs\experiment_7_24_data_fold_1_20251022_131153.json

📊 Epoch 1/10
--------------------------------------------------


KeyboardInterrupt: 

In [ ]:
# Alternative: Run pipeline for all days (if you want to compare across days)
# This will take significantly longer as it processes all three datasets

print("🌍 Option: Run Pipeline for All Days")
print("⚠️ This will take much longer as it processes all datasets")
print("💡 Only run this if you want cross-day comparison")

# Uncomment the lines below to run for all days
# all_results, comparison_df = run_pipeline_for_all_days(
#     output_base_dir="complete_pipeline_outputs",
#     fast_mode=FAST_MODE
# )

print("📝 Uncomment the lines above to run for all days")
print("🎯 This will process: 7_24_data, 10_19_data, and 11_10_data")

🌍 Option: Run Pipeline for All Days
⚠️ This will take much longer as it processes all datasets
💡 Only run this if you want cross-day comparison
📝 Uncomment the lines above to run for all days
🎯 This will process: 7_24_data, 10_19_data, and 11_10_data


## 13. Results Inspection

After running the pipeline, use these cells to inspect and analyze the results.

In [ ]:
# Inspect pipeline results (run this after the pipeline completes)
# This cell will display the final results and statistics

if 'results' in locals() and results is not None:
    print("🎯 Pipeline Results Summary:")
    print("=" * 50)
    
    stats = results['statistics']
    print(f"📊 Cross-Validation Statistics for {results['day_folder']}:")
    print(f"   Mean RMSE: {stats['mean_rmse']:.4f} ± {stats['std_rmse']:.4f}")
    print(f"   Mean MAE:  {stats['mean_mae']:.4f} ± {stats['std_mae']:.4f}")
    print(f"   Mean R²:   {stats['mean_r2']:.4f} ± {stats['std_r2']:.4f}")
    print(f"   Best Fold: {stats['best_fold']}")
    print(f"   Worst Fold: {stats['worst_fold']}")
    
    # Display fold metrics
    fold_metrics_df = pd.DataFrame(results['fold_metrics'])
    print(f"\n📈 Individual Fold Performance:")
    print(fold_metrics_df.round(4))
    
else:
    print("❌ No results found. Please run the pipeline first.")
    print("💡 Make sure to uncomment the execution line in the previous cell")

❌ No results found. Please run the pipeline first.
💡 Make sure to uncomment the execution line in the previous cell


In [ ]:
# Load and display saved results (if you want to examine results from a previous run)
import glob
import json

# Look for saved results
result_files = glob.glob(f"{OUTPUT_DIR}/results/cv_results_*.json")

if result_files:
    print(f"📁 Found {len(result_files)} result files:")
    for file in result_files:
        print(f"   - {file}")
    
    # Load the most recent results
    latest_file = max(result_files, key=os.path.getmtime)
    print(f"\n📊 Loading results from: {latest_file}")
    
    with open(latest_file, 'r') as f:
        saved_results = json.load(f)
    
    # Display summary
    stats = saved_results['statistics']
    print(f"\n🎯 Saved Results Summary for {saved_results['day_folder']}:")
    print(f"   Mean RMSE: {stats['mean_rmse']:.4f} ± {stats['std_rmse']:.4f}")
    print(f"   Mean MAE:  {stats['mean_mae']:.4f} ± {stats['std_mae']:.4f}")
    print(f"   Mean R²:   {stats['mean_r2']:.4f} ± {stats['std_r2']:.4f}")
    
else:
    print("❌ No saved results found.")
    print("💡 Run the pipeline first to generate results")

❌ No saved results found.
💡 Run the pipeline first to generate results


## 📝 Notes and Next Steps

**What this notebook does:**
1. ✅ Loads best hyperparameters from previous tuning
2. ✅ Trains CapsNet and LSTM with proper K-fold cross-validation
3. ✅ Extracts features from all trained models
4. ✅ Fuses CapsNet and LSTM features intelligently
5. ✅ Trains LightGBM on fused features
6. ✅ Provides comprehensive evaluation metrics
7. ✅ Creates publication-ready visualizations
8. ✅ Supports cross-day comparison analysis

**Key Features:**
- 🔄 **Proper Cross-Validation**: No data leakage between folds
- ⚡ **Fast Mode**: 3-fold CV for quick testing
- 📊 **Comprehensive Metrics**: RMSE, MAE, R² with confidence intervals
- 📈 **Rich Visualizations**: Fold comparison, feature importance, predictions vs actual
- 💾 **Result Persistence**: All results saved to disk
- 🌍 **Multi-Day Support**: Compare performance across different datasets

**Before Running:**
1. Ensure all data preprocessing is complete
2. Verify CapsNet and LSTM models are available
3. Check that hyperparameter tuning results exist
4. Confirm sufficient disk space for outputs

**After Running:**
1. Examine cross-validation statistics
2. Review feature importance plots
3. Analyze prediction quality across folds
4. Compare results across different days if applicable

**Configuration Tips:**
- Use `FAST_MODE=True` for initial testing (3-fold CV)
- Use `FAST_MODE=False` for final results (5-fold CV)
- Adjust `DAY_FOLDER` to process different datasets
- Check `OUTPUT_DIR` for all generated files

---
*This notebook provides a complete end-to-end pipeline for multi-modal air quality prediction using deep learning and ensemble methods.*